# 024 — CGC ESI Controller Test

Simple connect / activate / disconnect test for the CGC ESI-CTRL device.

In [1]:
import os
import logging
from pathlib import Path
from datetime import datetime

from devices.cgc.esi.esi import ESI

repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"024_cgc_esi_test_{timestamp}.log"

logger = logging.getLogger(f"024_cgc_esi_test_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

2026-04-09 14:41:35,495 - INFO - Logger initialized.


Log file: C:\Users\ESIBDlab\PycharmProjects\esibd_bs\debugging\logs\024_cgc_esi_test_20260409_144135.log


In [2]:
COM_PORT = 16  # adjust to the COM port the ESI controller is on

esi = ESI(device_id="esi_test", com=COM_PORT, logger=logger)

In [3]:
esi.connect()

2026-04-09 14:41:36,772 - INFO - esi_test - Connecting to ESI device esi_test on COM16
2026-04-09 14:41:36,940 - INFO - esi_test - Successfully connected to ESI device esi_test (baud rate: 230400)


True

In [4]:
esi.set_comspeed(115200)

(0, 115200)

In [5]:
esi.set_enable(True)

2026-04-09 14:41:39,517 - INFO - esi_test - Setting module enable to True
2026-04-09 14:41:39,519 - INFO - esi_test - Module enable set to True


0

## Live Current Monitor (web-based, opens in Chrome)

Bokeh server at ~1 Hz. Two stacked plots: HV current for module 2 and module 3 over time,
rolling window `ROLLING_WINDOW_MIN`. Runs in a background thread using `esi.thread_lock`,
so you can still call `set_hv_supply_target_output_voltage` etc. from other cells while it runs.

In [6]:
import time
import threading
import webbrowser
from datetime import datetime, timedelta

from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, DatetimeTickFormatter, Range1d
from bokeh.palettes import Category10
from bokeh.server.server import Server
from tornado.ioloop import IOLoop

POLL_INTERVAL_MS = 500   # 1 Hz
ROLLING_WINDOW_MIN = 1    # minutes visible
MAX_POINTS = 7200
BOKEH_PORT = 5008         # different from 023's 5007

MODULES = (2, 3)

# Suppress console logging during live monitoring (file logging continues)
console_handler.setLevel(logging.WARNING)


def make_document(doc):
    sources = {m: ColumnDataSource(data={"time": [], "i": []}) for m in MODULES}

    now = datetime.now()
    x_range = Range1d(start=now - timedelta(minutes=ROLLING_WINDOW_MIN), end=now)

    colors = Category10[3]
    figs = []
    for idx, m in enumerate(MODULES):
        p = figure(
            title=f"HV Module {m} — Output Current",
            x_axis_type="datetime",
            height=300, width=900,
            x_range=x_range,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            active_scroll="wheel_zoom",
        )
        p.line("time", "i", source=sources[m],
               line_width=2, color=colors[idx], legend_label=f"I mod{m}")
        p.yaxis.axis_label = "I [A]"
        p.xaxis.formatter = DatetimeTickFormatter(
            seconds="%H:%M:%S", minsec="%H:%M:%S", minutes="%H:%M:%S",
            hourmin="%H:%M:%S", hours="%H:%M:%S",
        )
        p.legend.location = "top_left"
        p.legend.click_policy = "hide"
        p.xaxis.axis_label = "Time" if idx == len(MODULES) - 1 else ""
        figs.append(p)

    doc.add_root(column(*figs, sizing_mode="stretch_width"))
    doc.title = "CGC ESI HV Current Monitor"

    def update():
        try:
            now = datetime.now()
            for m in MODULES:
                try:
                    with esi.thread_lock:
                        status, ok, i_a = esi.get_hv_supply_output_current(m)
                    if status == esi.NO_ERR:
                        sources[m].stream(
                            {"time": [now], "i": [i_a]},
                            rollover=MAX_POINTS,
                        )
                        logger.info(f"esi | mod{m} I={i_a*1e9:.2f} nA")
                except Exception as e:
                    logger.error(f"esi mod{m} current read failed: {e}")

            x_range.start = now - timedelta(minutes=ROLLING_WINDOW_MIN)
            x_range.end = now
        except Exception as e:
            logger.error(f"Plot update failed: {e}")

    doc.add_periodic_callback(update, POLL_INTERVAL_MS)


def _start_server():
    io_loop = IOLoop()
    io_loop.make_current()
    server = Server(
        {"/": make_document},
        port=BOKEH_PORT,
        io_loop=io_loop,
        allow_websocket_origin=[f"localhost:{BOKEH_PORT}"],
    )
    server.start()
    io_loop.start()


server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

time.sleep(1)
url = f"http://localhost:{BOKEH_PORT}/"
try:
    chrome = webbrowser.get("C:/Program Files/Google/Chrome/Application/chrome.exe %s")
    chrome.open(url)
    print(f"Opened Chrome at {url}")
except webbrowser.Error:
    webbrowser.open(url)
    print(f"Chrome not found, opened default browser at {url}")

print(f"Bokeh server running at {url}")
print("Server runs in background. You can set voltages from other cells while it polls.")

C:\Users\ESIBDlab\AppData\Local\Temp\ipykernel_8824\714528840.py:82: DeprecationWarning: make_current is deprecated; start the event loop first
  io_loop.make_current()


Opened Chrome at http://localhost:5008/
Bokeh server running at http://localhost:5008/
Server runs in background. You can set voltages from other cells while it polls.


In [9]:
console_handler.setLevel(logging.DEBUG)
logger.info("Live plot logging restored. Bokeh server thread will exit when kernel stops.")
print("Console logging restored. (Browser tab can be closed manually.)")

2026-04-09 12:01:43,111 - INFO - Live plot logging restored. Bokeh server thread will exit when kernel stops.


Console logging restored. (Browser tab can be closed manually.)


# Manual Control

In [7]:
esi.get_hv_supply_output_current(2)

(0, True, 6.3e-11)

In [8]:
esi.get_hv_supply_output_current(3)

(0, True, 8.3e-11)

In [9]:
esi.set_module_activation_state(3, True)

0

In [10]:
esi.set_module_activation_state(2, True)

0

In [11]:
esi.get_hv_supply_output_voltage(2)

(0, True, -0.052)

In [12]:
esi.get_hv_supply_output_voltage(3)

(0, True, -0.047)

In [35]:
esi.set_hv_supply_target_output_voltage(2, 0.0)

0

In [37]:
esi.set_hv_supply_target_output_voltage(3, 1500.0)

0

In [38]:
esi.set_hv_supply_target_output_voltage(2, 0.0)

0

In [39]:
esi.set_hv_supply_target_output_voltage(3, 0.0)

0

In [40]:
esi.set_module_activation_state(2, False)
esi.set_module_activation_state(3, False)

0

In [41]:
esi.set_enable(False)

0

In [42]:
esi.get_enable()

(0, False)

In [1]:
esi.disconnect()

NameError: name 'esi' is not defined